# Interactive plotting with `ehtim.plotting.interactive`

A Plotly-based companion to the matplotlib `comp_plots` / `summary_plots`
defaults. Same data, more interactivity: click legends to toggle
baselines, drag to zoom, hover for per-point hovertext, and use the
Color toolbar to highlight individual traces.

This notebook walks through every public function (`plot_bl`, `plotall`,
`plot_gains`, `dashboard`) plus saving figures to standalone HTML.

Requires `plotly` (in the `dev` extra, or `pip install plotly`).

## Setup

Load a sample uvfits + image; we use these throughout.

**Display convention.** Every plotting function returns a Plotly
Figure. To get the Color toolbar in Jupyter, pass `show=False` and then
call `interactive.display(fig)` (this injects the toolbar JS). Plain
`fig` auto-display works too but shows the figure without the toolbar.

In [ ]:
import ehtim as eh
from ehtim.plotting import interactive

obs = eh.obsdata.load_uvfits('../data/sample.uvfits')
im = eh.image.load_txt('../models/avery_sgra_eofn.txt')
print(obs)


## `plot_bl` - one baseline at a time

Plot any visibility field for a single (site1, site2) pair against time.

In [ ]:
fig = interactive.plot_bl(obs, 'ALMA', 'LMT', 'amp', show=False)
interactive.display(fig)


Other useful fields: `'phase'`, `'snr'`, `'uvdist'`, `'sigma'`.
Polarimetric obs also expose per-hand visibilities (`'rrvis'`, `'llvis'`,
`'rlvis'`, `'lrvis'`).

In [ ]:
fig = interactive.plot_bl(obs, 'ALMA', 'SMT', 'phase', show=False)
interactive.display(fig)


## `plotall` - any field against any other

The general two-field scatter. `tag_bl=True` (default) splits the data
into per-baseline traces; `tag_bl=False` pools everything into one
trace.

In [ ]:
# Radplot: amplitude vs baseline length, one trace per baseline.
fig = interactive.plotall(obs, 'uvdist', 'amp', show=False)
interactive.display(fig)


### UV coverage

`plotall(obs, 'u', 'v')` is the UV-coverage view. Locked to a square
aspect, symmetric range driven by max(|u|, |v|), and axis values
auto-scaled to Gλ or Mλ. Pass `conj=True` to overlay the Hermitian
conjugate half.

In [ ]:
fig = interactive.plotall(obs, 'u', 'v', conj=True, show=False)
interactive.display(fig)


## `plot_gains` - caltable gain time series

Drop in a calibration table from `self_cal` or `network_cal` to see
per-site amplitude or phase gains. `pol='R'` or `'L'` picks the channel;
`gain_type` selects amplitude vs phase.

(TODO: schema-coupled - `R`/`L` become `pol1`/`pol2` once the mixed-pol
cal table lands.)

In [ ]:
from ehtim.calibrating.self_cal import self_cal

caltable = self_cal(
    obs, im, method='both', caltable=True,
    processes=2, show_solution=False,
)
fig = interactive.plot_gains(caltable, gain_type='amp', pol='R', show=False)
interactive.display(fig)


## `dashboard` - image + data + gains + D-terms in one view

A 2x2 dashboard summarizing a reconstruction.

- **Top left** - Stokes-I image in μas RA/Dec offsets. EVPA ticks
  colored by fractional polarization with a rainbow colorbar adjacent
  to the intensity colorbar. The "Toggle polarization" button hides
  or shows the overlay.
- **Top right** - data-product dropdown (12 options): amplitude / Re(vis)
  / phase / amplitude residual / closure phase (and residual) vs time /
  log closure amp (and residual) vs time / closure quantities vs
  triangle or quadrangle area / |m_breve| (and residual). Data and model
  traces toggle independently in the legend.
- **Bottom left** - per-site gain time series with a dropdown for
  amp R, amp L, phase R, phase L.
- **Bottom right** - D-terms in the complex plane with two legends:
  one per-site (toggles both R and L for that site), one showing the
  R-circle / L-square symbol convention.

In [ ]:
# Inject linear pol so the dashboard's EVPA overlay has something to show,
# and synthetic D-terms so the bottom-right panel isn't all zero.
import numpy as np

im_pol = im.copy()
I2d = im.imvec.reshape(im.ydim, im.xdim)
im_pol.add_qu(0.1 * I2d, 0.05 * I2d)

ct = caltable.copy()
rng = np.random.default_rng(0)
n = len(ct.tarr)
ct.tarr['dr'][:] = 0.05 * (rng.standard_normal(n) + 1j * rng.standard_normal(n))
ct.tarr['dl'][:] = 0.05 * (rng.standard_normal(n) + 1j * rng.standard_normal(n))

fig = interactive.dashboard(im_pol, obs, ct, show=False)
interactive.display(fig)


## Color toolbar

`plot_bl(tag_bl=True)`, `plotall(tag_bl=True)`, and the dashboard panels
add a three-button toolbar above the figure:

- **Color** - click to enter Color mode. While on, clicking a legend
  entry paints (or un-paints) that single trace instead of plotly's
  default hide.
- **Color all** - paint every trace at once.
- **Show all / reset** - return everything to gray + visible.

Color mode is independent from plotly's built-in hide behavior - turn
Color mode off and legend clicks revert to the default toggle.

## Saving to HTML

`fig.write_html(path)` works directly (the returned figure is patched
so the toolbar JS is embedded automatically), or use
`interactive.write_html(fig, path)`. The PNG button on the plotly
modebar saves at 3x scale (≈ 300 dpi) and sizes the canvas to fit
the legend.

In [ ]:
fig = interactive.plotall(obs, 'u', 'v', show=False)
fig.write_html('/tmp/uv_coverage.html', include_plotlyjs='cdn')
print('saved /tmp/uv_coverage.html')
